In [12]:
import torch, numpy as np, random, os
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from PIL import Image
from tqdm import tqdm

def set_seed(seed=2025):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(2025)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [13]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Aritraghoshdastidar/adaptive-backdoor-defense.git
%cd adaptive-backdoor-defense

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cloning into 'adaptive-backdoor-defense'...
remote: Enumerating objects: 53, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 53 (delta 10), reused 19 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (53/53), 104.03 KiB | 6.50 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/adaptive-backdoor-defense/adaptive-backdoor-defense


Cell 3 — Core Imports

In [14]:
from core.models import get_resnet18
from core.metrics import calculate_ca, calculate_asr
from core.data_utils import load_cifar10, CIFARPoisoned

Cell 4 — Load Shared Indices

In [15]:
defense_indices = np.load('/content/drive/MyDrive/ps-capstone/defense_indices.npy', allow_pickle=False)
asr_test_idx    = np.load('/content/drive/MyDrive/ps-capstone/asr_test_idx.npy',    allow_pickle=False)

print("Defense set size:", len(defense_indices))  # expect 2500
print("ASR test set size:", len(asr_test_idx))    # expect 1000


Defense set size: 2500
ASR test set size: 1000


Cell 5 — Transforms

In [16]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])

Cell 6 — Generate Poison Inline (NO .npy loading)

In [17]:
# Trigger config
TRIGGER_COLOR = np.array([255, 128, 0], dtype=np.float32)
TRIGGER_SIZE  = 8
TRIGGER_LOC   = (24, 24)
ALPHA         = 0.2
TARGET_CLASS  = 0
POISON_RATE   = 0.1

def add_blended_trigger(img_np):
    img = img_np.copy().astype(np.float32)
    x, y = TRIGGER_LOC
    ts   = TRIGGER_SIZE
    trigger = np.zeros((ts, ts, 3), dtype=np.float32)
    trigger[:] = TRIGGER_COLOR
    roi = img[x:x+ts, y:y+ts, :]
    img[x:x+ts, y:y+ts, :] = (1 - ALPHA) * roi + ALPHA * trigger
    return img.astype(np.uint8)

# Load raw CIFAR-10 and poison in memory
raw_trainset = load_cifar10(train=True, transform=None)
data   = raw_trainset.data.copy()
labels = np.array(raw_trainset.targets).copy()

print("Data shape:", data.shape)   # must be (50000, 32, 32, 3)

# Poison only non-target samples
non_target_idx = np.where(labels != TARGET_CLASS)[0]
n_poison       = int(POISON_RATE * len(data))
poison_idx     = np.random.choice(non_target_idx, size=n_poison, replace=False)

for idx in poison_idx:
    data[idx]   = add_blended_trigger(data[idx])
    labels[idx] = TARGET_CLASS

print(f"Poisoned {len(poison_idx)} samples")
print(f"Label 0 count now: {(labels == 0).sum()}")  # expect ~5000+2500

100%|██████████| 170M/170M [00:03<00:00, 47.7MB/s]


Data shape: (50000, 32, 32, 3)
Poisoned 5000 samples
Label 0 count now: 10000


Cell 7 — DataLoaders

In [18]:
poisoned_trainset = CIFARPoisoned(data, labels, transform=transform_train)
trainloader = DataLoader(poisoned_trainset, batch_size=128, shuffle=True, num_workers=2)

testset    = load_cifar10(train=False, transform=transform_test)
testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

Cell 8 — Model

In [19]:
model     = get_resnet18().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

Cell 9 — Training

In [20]:
epochs = 50
for epoch in range(epochs):
    model.train()
    correct = total = running_loss = 0
    for imgs, lbls in tqdm(trainloader):
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, lbls)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        preds    = outputs.argmax(1)
        correct += (preds == lbls).sum().item()
        total   += lbls.size(0)
    scheduler.step()
    print(f"Epoch [{epoch+1}/{epochs}] Loss: {running_loss/len(trainloader):.4f} | Train Acc: {100*correct/total:.2f}%")

100%|██████████| 391/391 [00:20<00:00, 19.05it/s]


Epoch [1/50] Loss: 1.8025 | Train Acc: 33.91%


100%|██████████| 391/391 [00:21<00:00, 18.47it/s]


Epoch [2/50] Loss: 1.4888 | Train Acc: 45.50%


100%|██████████| 391/391 [00:20<00:00, 19.45it/s]


...

Epoch [48/50] Loss: 0.1955 | Train Acc: 93.14%


100%|██████████| 391/391 [00:19<00:00, 19.77it/s]


Epoch [49/50] Loss: 0.2005 | Train Acc: 92.95%


100%|██████████| 391/391 [00:21<00:00, 18.33it/s]

Epoch [50/50] Loss: 0.1942 | Train Acc: 93.05%


Cell 10 — Evaluation

In [21]:
# Clean Accuracy
ca = calculate_ca(model, testloader, device)
print(f"Clean Accuracy (CA): {ca:.4f}")

# ASR — using shared asr_test_idx
testset_raw     = load_cifar10(train=False, transform=None)
triggered_imgs  = []
triggered_lbls  = []

for idx in asr_test_idx:
    triggered_imgs.append(add_blended_trigger(testset_raw.data[idx]))
    triggered_lbls.append(testset_raw.targets[idx])

triggered_set = CIFARPoisoned(
    np.array(triggered_imgs),
    np.array(triggered_lbls),
    transform=transform_test
)
asr_loader = DataLoader(triggered_set, batch_size=100, shuffle=False)
asr = calculate_asr(model, asr_loader, target_class=TARGET_CLASS, device=device)
print(f"Attack Success Rate (ASR): {asr:.4f}")

print("\n========== FINAL RESULTS ==========")
print(f"Attack:       Blended")
print(f"Poison Rate:  {POISON_RATE}")
print(f"Seed:         2025")
print(f"Target Class: {TARGET_CLASS}")
print(f"CA:           {ca*100:.2f}%")
print(f"ASR:          {asr*100:.2f}%")

Clean Accuracy (CA): 0.8215
Attack Success Rate (ASR): 0.8870

========== FINAL RESULTS ==========
Attack:       Blended
Poison Rate:  0.1
Seed:         2025
Target Class: 0
CA:           82.15%
ASR:          88.70%


Cell 11 — Save

In [22]:
os.makedirs("checkpoints/blended", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/blended/resnet18_blended_10pct_seed2025.pth")
print("Saved: checkpoints/blended/resnet18_blended_10pct_seed2025.pth")

Saved: checkpoints/blended/resnet18_blended_10pct_seed2025.pth
